# llmfp on Colab

Collects fingerprints for several Hugging Face models on one GPU and analyzes them together. It clones the repository named in `REPO_URL`. To use a local copy instead, set `REPO_URL` to an empty string and upload a zip of the `llm-fingerprint` folder when prompted.

In [ ]:
REPO_URL = "https://github.com/dwtrott/llm_psychometric_test.git"   # set to "" to upload a zip instead
import os, subprocess, zipfile
if REPO_URL:
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, "llmfp_repo"], check=True)
else:
    from google.colab import files
    name = next(iter(files.upload()))
    zipfile.ZipFile(name).extractall("llmfp_repo")
root = next(os.path.join(d, "") for d, _, f in os.walk("llmfp_repo") if "pyproject.toml" in f)
%cd $root
!pip -q install -e ".[hf]"
!python tests/test_llmfp.py

Each entry is `name: Hugging Face id`. Include known relatives, such as a quantized copy or a fine-tune of one of the bases, and declare them in `GROUPS`. Gated models need `huggingface-cli login` first.

In [ ]:
MODELS = {
    "qwen2.5-7b": "Qwen/Qwen2.5-7B-Instruct",
    "qwen2.5-3b": "Qwen/Qwen2.5-3B-Instruct",
    "phi-3.5-mini": "microsoft/Phi-3.5-mini-instruct",
    "smollm2-1.7b": "HuggingFaceTB/SmolLM2-1.7B-Instruct",
}
FOUR_BIT_COPIES = ["qwen2.5-7b"]      # also collected with 4-bit quantization as a known relative
GROUPS = {f"{name}-4bit": name for name in FOUR_BIT_COPIES}

In [ ]:
import json, subprocess, sys
os.makedirs("obs", exist_ok=True)
jobs = [(name, hf_id, []) for name, hf_id in MODELS.items()]
jobs += [(f"{name}-4bit", MODELS[name], ["--load-in-4bit"]) for name in FOUR_BIT_COPIES]
if FOUR_BIT_COPIES:
    !pip -q install bitsandbytes
for name, hf_id, extra in jobs:      # one process per model so that GPU memory is released
    subprocess.run([sys.executable, "-m", "llmfp", "run-hf", hf_id, "--name", name,
                    "--out", f"obs/{name}.jsonl", "--dtype", "bfloat16", "--context-set", "all", *extra], check=True)
json.dump(GROUPS, open("groups.json", "w"))

In [ ]:
!python -m llmfp analyze obs/*.jsonl --out report --groups groups.json
import shutil
shutil.make_archive("llmfp_results", "zip", ".", "report")
shutil.make_archive("llmfp_observations", "zip", ".", "obs")
from google.colab import files
files.download("llmfp_results.zip"); files.download("llmfp_observations.zip")